In this Colab, we will create a classification model to determine if an image is a dog or a cat.

In [2]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import os
import numpy as np

# Optional: Check TensorFlow version just to be sure
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.18.0


Now we are going to pull in our dogs and cats images and automatically label them based off of the folder they are in. We will also resize the images to be a consistent size and rescale the pixel values which are usually from 0 - 255 to 0 - 1 which will help the neural network train more effectively.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
training_set = '/content/drive/MyDrive/Data/classification_data/training_set/training_set'

Create an ImageDataGenerator and split our training data so that 20% will be used for validation

The ImageDataGenerator handles:

  Splitting your data into training and validation (using subset='training' or subset='validation'),
  
  Applying any specified preprocessing or augmentation (rotation, flipping, etc.),

And generating the images in batches, so you don’t have to load everything into memory at once.

In [5]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Creates a data generator with augmentation + rescaling for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # 20% goes to validation
)

# Defines the training subset (80%)
train_data = train_datagen.flow_from_directory(
    training_set,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

# Defines the validation subset (20%)
val_data = train_datagen.flow_from_directory(
    training_set,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

Found 6415 images belonging to 2 classes.
Found 1603 images belonging to 2 classes.


Here we are defining our CNN Model which is a Sequential Model with convolutional layers, pooling layers, and fully connected layers at the en.

This model has an input layer with our 150x150 pixel images with a batch size of 32. There are 3 hidden layers the first with 32 nodes, second with 64, and the last with 128. They are all using the relu activation function.

We are also flattening our layers right before the last hidden layer to convert it into a single fully connected vector mapping to the output layer which is ideal for relationship understanding.

In [6]:
import tensorflow as tf
from tensorflow.keras import layers, models

model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Now we will compile the model using the adam optimizer which is what goes and actually changes the weights and biases after backpropagation calculates the gradients and we calculate gradient descent which essentially tweaks all of the weights and biases in a way that will most rapidly decrease the loss.

In [7]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

We are going to add an early stoppage callback to stop training automatiucally if the validation loss stops improving.

In [11]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',      # we monitor the validation loss
    patience=10,              # wait 10 epochs after the last improvement
    restore_best_weights=True  # revert to the best model weights
)

In [12]:
history = model.fit(
    train_data,
    epochs=200,
    validation_data=val_data,
    callbacks=[early_stop],
    verbose=2
)

Epoch 1/200
201/201 - 74s - 369ms/step - accuracy: 0.8444 - loss: 0.3522 - val_accuracy: 0.8347 - val_loss: 0.3848
Epoch 2/200
201/201 - 75s - 373ms/step - accuracy: 0.8493 - loss: 0.3413 - val_accuracy: 0.8172 - val_loss: 0.4070
Epoch 3/200
201/201 - 71s - 356ms/step - accuracy: 0.8494 - loss: 0.3370 - val_accuracy: 0.8129 - val_loss: 0.3963
Epoch 4/200
201/201 - 73s - 362ms/step - accuracy: 0.8560 - loss: 0.3266 - val_accuracy: 0.8060 - val_loss: 0.4101
Epoch 5/200
201/201 - 73s - 361ms/step - accuracy: 0.8528 - loss: 0.3318 - val_accuracy: 0.8334 - val_loss: 0.3679
Epoch 6/200
201/201 - 73s - 364ms/step - accuracy: 0.8634 - loss: 0.3158 - val_accuracy: 0.8428 - val_loss: 0.3603
Epoch 7/200
201/201 - 71s - 356ms/step - accuracy: 0.8650 - loss: 0.3094 - val_accuracy: 0.8278 - val_loss: 0.3906
Epoch 8/200
201/201 - 73s - 364ms/step - accuracy: 0.8645 - loss: 0.3087 - val_accuracy: 0.8328 - val_loss: 0.3536
Epoch 9/200
201/201 - 72s - 356ms/step - accuracy: 0.8673 - loss: 0.3065 - val_a

Now we will evaluate our model on the test set to see how it performs

In [15]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

test_data = '/content/drive/MyDrive/Data/classification_data/test_set/test_set'

# Typically just rescale for test; no augmentations
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_directory(
    directory=test_data,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

test_loss, test_acc = model.evaluate(test_generator)
print("Test Loss:", test_loss)
print("Test Accuracy:", test_acc)

Found 2023 images belonging to 2 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


64/64 ━━━━━━━━━━━━━━━━━━━━ 552s 9s/step - accuracy: 0.8587 - loss: 0.3967
Test Loss: 0.3527226746082306
Test Accuracy: 0.8719723224639893


In [16]:
model.save('dog_cat_classifier.keras')

If I want to use this model later for any reason I can just use the following command: from tensorflow.keras.models import load_model
loaded_model = load_model('dog_cat_classifier.keras')